In [1]:
import tkinter as tk
from tkinter import messagebox

# =====================================================================
# 1. CAPA DE IMPLEMENTACIÓN (LÓGICA DEL BST)
# =====================================================================
class NodoCancion:
    def __init__(self, duracion):
        self.duracion = duracion  
        self.izq = None           
        self.der = None           

class ArbolCanciones:
    def __init__(self):
        self.raiz = None

    def insertar(self, duracion):
        self.raiz = self._insertar_recursivo(self.raiz, duracion)

    def _insertar_recursivo(self, nodo, duracion):
        if nodo is None:
            return NodoCancion(duracion)
        if duracion < nodo.duracion:
            nodo.izq = self._insertar_recursivo(nodo.izq, duracion)
        elif duracion > nodo.duracion:
            nodo.der = self._insertar_recursivo(nodo.der, duracion)
        return nodo

    def calcular_tiempo_total(self):
        return self._calcular_tiempo_total_recursivo(self.raiz)

    def _calcular_tiempo_total_recursivo(self, nodo):
        if nodo is None:
            return 0
        return nodo.duracion + self._calcular_tiempo_total_recursivo(nodo.izq) + self._calcular_tiempo_total_recursivo(nodo.der)

    def recomendar_cancion_perfecta(self, segundos_disponibles):
        if self.raiz is None:
            return None
        return self._recomendar_eficiente(self.raiz, segundos_disponibles, self.raiz.duracion)

    def _recomendar_eficiente(self, nodo, objetivo, mejor_hasta_ahora):
        if nodo is None:
            return mejor_hasta_ahora
        if nodo.duracion == objetivo:
            return nodo.duracion
        if abs(nodo.duracion - objetivo) < abs(mejor_hasta_ahora - objetivo):
            mejor_hasta_ahora = nodo.duracion
        if objetivo < nodo.duracion:
            return self._recomendar_eficiente(nodo.izq, objetivo, mejor_hasta_ahora)
        else:
            return self._recomendar_eficiente(nodo.der, objetivo, mejor_hasta_ahora)

    def filtrar_canciones_cortas(self, duracion_minima):
        self.raiz = self._filtrar_recursivo(self.raiz, duracion_minima)

    def _filtrar_recursivo(self, nodo, min_duracion):
        if nodo is None:
            return None
        nodo.izq = self._filtrar_recursivo(nodo.izq, min_duracion)
        nodo.der = self._filtrar_recursivo(nodo.der, min_duracion)
        if nodo.duracion < min_duracion:
            return nodo.der
        return nodo

    def contar_canciones(self):
        return self._contar_canciones_recursivo(self.raiz)

    def _contar_canciones_recursivo(self, nodo):
        if nodo is None:
            return 0
        return 1 + self._contar_canciones_recursivo(nodo.izq) + self._contar_canciones_recursivo(nodo.der)

    def calcular_duracion_promedio(self):
        total_tiempo = self.calcular_tiempo_total()
        total_canciones = self.contar_canciones()
        if total_canciones == 0:
            return 0
        return total_tiempo / total_canciones

 
    def obtener_estructura_texto(self):
        lineas = []
        self._escribir_estructura(self.raiz, 0, "Raíz", lineas)
        if not lineas:
            return "🌳 El árbol de reproducción está completamente vacío.\n✍️ Ingresa duraciones en la sección superior para construirlo."
        
        estadisticas = (
            f"📊 ESTADÍSTICAS DEL CATÁLOGO:\n"
            f" ⏱️ Tiempo Total: {self.calcular_tiempo_total()} seg\n"
            f" 🎵 Total Canciones: {self.contar_canciones()}\n"
            f" 📈 Duración Promedio: {self.calcular_duracion_promedio():.1f} seg [Función Propia]\n"
            f"--------------------------------------------------\n"
            f"MAPA DE PUNTEROS EN MEMORIA (BST):\n"
        )
        return estadisticas + "\n".join(lineas)

    def _escribir_estructura(self, nodo, nivel, relacion, lineas):
        if nodo is not None:
            indentacion = "    " * nivel
            lineas.append(f"{indentacion}└── [{relacion}] Canción: {nodo.duracion} seg")
            if nodo.izq or nodo.der:
                if nodo.izq:
                    self._escribir_estructura(nodo.izq, nivel + 1, "Izq", lineas)
                else:
                    lineas.append(f"{indentacion}    └── [Izq] None")
                if nodo.der:
                    self._escribir_estructura(nodo.der, nivel + 1, "Der", lineas)
                else:
                    lineas.append(f"{indentacion}    └── [Der] None")


# =====================================================================
# 2. CAPA DE INTERFAZ GRÁFICA 
# =====================================================================
class AppMusicalBST:
    def __init__(self, ventana):
        self.ventana = ventana
        self.ventana.title("Estructura de Datos - Árboles Binarios (BST)")
        self.ventana.geometry("600x650")
        self.ventana.configure(bg="#F0F0F0")

        self.arbol = ArbolCanciones()

        lbl_titulo = tk.Label(ventana, text="🎵 MOTOR DE RECOMENDACIÓN MUSICAL (BST) 🎵", 
                              font=("Arial", 12, "bold"), bg="#F0F0F0", fg="#333333")
        lbl_titulo.pack(pady=10)

        # 1. INSERTAR CANCIÓN
        frame_ins = tk.LabelFrame(ventana, text=" 1. Registrar Nueva Canción ", font=("Arial", 10, "bold"), bg="#F0F0F0", padx=10, pady=5)
        frame_ins.pack(fill="x", padx=15, pady=5)
        
        tk.Label(frame_ins, text="Duración (segundos):", bg="#F0F0F0").pack(side="left", padx=5)
        self.entry_duracion = tk.Entry(frame_ins, width=15)
        self.entry_duracion.pack(side="left", padx=5)
        
        btn_ins = tk.Button(frame_ins, text="Añadir al Árbol", command=self.op_insertar, bg="#E1E1E1", width=15)
        btn_ins.pack(side="right", padx=5)

        # 2. RECOMENDACIÓN INTELIGENTE
        frame_rec = tk.LabelFrame(ventana, text=" 2. Buscar Canción Ideal (Sugerencia) ", font=("Arial", 10, "bold"), bg="#F0F0F0", padx=10, pady=5)
        frame_rec.pack(fill="x", padx=15, pady=5)
        
        tk.Label(frame_rec, text="Tiempo disponible (seg):", bg="#F0F0F0").pack(side="left", padx=5)
        self.entry_objetivo = tk.Entry(frame_rec, width=15)
        self.entry_objetivo.pack(side="left", padx=5)
        
        btn_rec = tk.Button(frame_rec, text="Calcular Sugerencia", command=self.op_recomendar, bg="#E1E1E1", width=15)
        btn_rec.pack(side="right", padx=5)

        # 3. FILTRAR AUDIO BASURA
        frame_fil = tk.LabelFrame(ventana, text=" 3. Filtrar Catálogo (Limpieza) ", font=("Arial", 10, "bold"), bg="#F0F0F0", padx=10, pady=5)
        frame_fil.pack(fill="x", padx=15, pady=5)
        
        tk.Label(frame_fil, text="Duración Mínima (seg):", bg="#F0F0F0").pack(side="left", padx=5)
        self.entry_minimo = tk.Entry(frame_fil, width=15)
        self.entry_minimo.pack(side="left", padx=5)
        
        btn_fil = tk.Button(frame_fil, text="Eliminar Cortas", command=self.op_filtrar, bg="#E1E1E1", width=15)
        btn_fil.pack(side="right", padx=5)

        # PANEL INFERIOR: CONSOLA VISUAL
        lbl_vista = tk.Label(ventana, text="ESTADO DEL ÁRBOL EN MEMORIA SIMULADA", font=("Arial", 9, "bold"), bg="#F0F0F0", fg="#555555")
        lbl_vista.pack(pady=(15, 0))

        self.txt_pantalla = tk.Text(ventana, bg="white", fg="black", font=("Courier New", 10), bd=2, relief="groove")
        self.txt_pantalla.pack(fill="both", expand=True, padx=15, pady=10)

        self.actualizar_pantalla()

    def actualizar_pantalla(self):
        contenido = self.arbol.obtener_estructura_texto()
        self.txt_pantalla.config(state=tk.NORMAL)
        self.txt_pantalla.delete("1.0", tk.END)
        self.txt_pantalla.insert(tk.END, contenido)
        self.txt_pantalla.config(state=tk.DISABLED)

    def op_insertar(self):
        try:
            valor = int(self.entry_duracion.get().strip())
            if valor <= 0:
                raise ValueError
            self.arbol.insertar(valor)
            messagebox.showinfo("Operación 1", f"✅ Canción de {valor} segundos añadida correctamente al árbol.")
            self.entry_duracion.delete(0, tk.END)
            self.actualizar_pantalla()
        except ValueError:
            messagebox.showerror("Error", "⚠️ La duración debe ser un número entero positivo.")

    def op_recomendar(self):
        try:
            objetivo = int(self.entry_objetivo.get().strip())
            if objetivo <= 0:
                raise ValueError
            
            sugerencia = self.arbol.recomendar_cancion_perfecta(objetivo)
            if sugerencia is not None:
                messagebox.showinfo("Operación 2", f"🎯 Sugerencia:\nLa mejor opción guardada es la canción de {sugerencia} segundos.")
            else:
                messagebox.showwarning("Operación 2", "⚠️ El catálogo está vacío.")
            self.entry_objetivo.delete(0, tk.END)
        except ValueError:
            messagebox.showerror("Error", "⚠️ Ingresa un número entero positivo.")

    def op_filtrar(self):
        try:
            limite = int(self.entry_minimo.get().strip())
            if limite <= 0:
                raise ValueError
            
            self.arbol.filtrar_canciones_cortas(limite)
            messagebox.showinfo("Operación 3", f"🧹 Se han removido los audios menores a {limite} segundos.")
            self.entry_minimo.delete(0, tk.END)
            self.actualizar_pantalla()
        except ValueError:
            messagebox.showerror("Error", "⚠️ El límite debe ser un número entero positivo.")

# --- ARRANQUE SEGURO ---
if __name__ == "__main__":
    try:
        if 'ventana' in globals():
            ventana.destroy()
    except:
        pass
        
    ventana = tk.Tk()
    app = AppMusicalBST(ventana)
    
    ventana.lift()
    ventana.attributes('-topmost', True)
    ventana.after(1, lambda: ventana.attributes('-topmost', False))
    
    ventana.mainloop()